# Постачання

In [24]:
pip install Office365-REST-Python-Client gspread gspread-dataframe pandas xmltodict google-auth python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [25]:
import os
import tempfile
import pandas as pd
import xmltodict
import requests
import gspread
import gspread.utils
from gspread_dataframe import set_with_dataframe

# Авторизація через сервіс-акаунт Google (без інтерактивного логіну — працює headless)
from google.oauth2.service_account import Credentials as ServiceAccountCredentials

# Шлях до JSON-ключа сервіс-акаунта. Файл лежить ПОЗА цим репозиторієм/ноутбуком — не комітити нікуди.
# Таблицю треба розшарити (Editor) на client_email з цього ключа.
SERVICE_ACCOUNT_FILE = r"C:\Users\Administrator\.secrets\google\lbc-infrastructure-kestra-sheets-bot.json"
SCOPES = ["https://www.googleapis.com/auth/spreadsheets"]

def get_google_creds():
    return ServiceAccountCredentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=SCOPES)

# ==========================================
# ⚙️ НАЛАШТУВАННЯ MICROSOFT GRAPH API
# ==========================================
TENANT_ID = "vyriyllc.onmicrosoft.com"
CLIENT_ID = "7dd77777-084a-4585-a1fd-5f37696b4f9f"
# .env лежить поруч із ноутбуком (Colab/.env), у git не потрапляє
from dotenv import load_dotenv
load_dotenv(r"C:\Users\Administrator\Documents\DWH\Kestra YAML\Colab\.env")
CLIENT_SECRET = os.environ["MS_GRAPH_CLIENT_SECRET"]

SITE_ID = "vyriyllc.sharepoint.com,1cea33e3-a302-4a52-a75a-ff85fde14516,af520fba-93b7-4556-954d-decd4a92a686"
ITEM_PATH = "/LatestXML/Postach latest/postachannya_latest.xml"

# Список колонок, які треба зробити числами
NUMERIC_COLUMNS = [
    "product_price_base_u",
    "product_count",
    "product_reserved",
    "product_price_base"
]

# ==========================================
# ⚙️ НАЛАШТУВАННЯ GOOGLE SHEETS
# ==========================================
GS_SHEET_URL = "https://docs.google.com/spreadsheets/d/1yHi1I3F9qhm-V1AdqaklhwO8DZGjlqEkHwqDt35QzH8/edit?gid=1733056911#gid=1733056911"
GS_WORKSHEET_NAME = "Постачання (всі)"

def get_graph_token():
    """Отримання токена доступу від Azure AD"""
    print("🔑 Отримуємо токен доступу від Azure AD...")
    token_url = f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token"
    token_data = {
        'grant_type': 'client_credentials',
        'client_id': CLIENT_ID,
        'client_secret': CLIENT_SECRET,
        'scope': 'https://graph.microsoft.com/.default'
    }
    res = requests.post(token_url, data=token_data)
    if res.status_code != 200:
        print(f"❌ Помилка авторизації:\n{res.text}")
        return None
    return res.json().get('access_token')

def download_xml_via_graph(token):
    """Завантаження файлу безпосередньо через Microsoft Graph API"""
    file_name = os.path.basename(ITEM_PATH)
    print(f"⏳ Стукаємось до файлу: {file_name}...")

    download_url = f"https://graph.microsoft.com/v1.0/sites/{SITE_ID}/drive/root:{ITEM_PATH}:/content"
    headers = {'Authorization': f'Bearer {token}'}

    file_res = requests.get(download_url, headers=headers)
    if file_res.status_code != 200:
        print(f"❌ Помилка завантаження файлу:\n{file_res.text}")
        return None, None

    temp_dir = tempfile.mkdtemp()
    download_path = os.path.join(temp_dir, file_name)

    with open(download_path, "wb") as f:
        f.write(file_res.content)

    print(f"📥 Успішно завантажено: {file_name}")
    return download_path, file_name

def parse_xml_to_dataframe(file_path):
    """Інтелектуальний парсинг XML у плоску таблицю з розгортанням вкладених списків"""
    all_records = []

    def find_data_records(data):
        found_lists = []
        def traverse(node, path):
            if isinstance(node, dict):
                for key, value in node.items():
                    traverse(value, path + [key])
            elif isinstance(node, list):
                if len(node) > 0 and isinstance(node[0], dict):
                    found_lists.append((path, node))
        traverse(data, [])

        if found_lists:
            best_path, best_data = max(found_lists, key=lambda x: len(x[1]))
            print(f"🤖 Авто-скан: знайдено таблицю за шляхом [{' ➡️ '.join(best_path)}] (Рядків: {len(best_data)})")
            return best_data

        def get_deepest(node, path):
            if isinstance(node, dict) and len(node) == 1:
                k = list(node.keys())[0]
                if isinstance(node[k], (dict, list)):
                    return get_deepest(node[k], path + [k])
            return path, node

        path, node = get_deepest(data, [])
        if isinstance(node, dict):
            return [node]
        elif isinstance(node, list):
            return node
        return [data]

    with open(file_path, 'r', encoding='utf-8') as f:
        xml_content = f.read()

    doc = xmltodict.parse(xml_content)
    records = find_data_records(doc)
    all_records.extend(records)
    print(f"🔄 Опрацьовано файл: {os.path.basename(file_path)}")

    # Початкове розгортання
    df = pd.json_normalize(all_records)

    # --- 🪄 МАГІЯ ДЛЯ РОЗГОРТАННЯ СПИСКІВ (ТОВАРІВ) ---
    list_columns = [col for col in df.columns if df[col].apply(lambda x: isinstance(x, list)).any()]
    for col in list_columns:
        print(f"📦 Розгортаємо вкладений список у колонці '{col}'...")
        df = df.explode(col).reset_index(drop=True)
        if df[col].apply(lambda x: isinstance(x, dict)).any():
            col_df = pd.json_normalize(df[col].apply(lambda x: x if isinstance(x, dict) else {}))
            col_df = col_df.add_prefix(f"{col}_")
            df = df.drop(columns=[col]).join(col_df)

    # --- 🔢 КОНВЕРТАЦІЯ ТИПІВ ДАНИХ (ДЕСЯТКОВІ ЧИСЛА) ---
    print("🛠 Конвертуємо типи колонок...")
    for col in NUMERIC_COLUMNS:
        if col in df.columns:
            # Замінюємо кому на крапку і переводимо у float (десяткове число)
            df[col] = df[col].astype(str).str.replace(',', '.').str.replace(' ', '')
            df[col] = pd.to_numeric(df[col], errors='coerce')

    return df

def upload_to_google_sheets(df):
    """Запис DataFrame у Google Sheets через сервіс-акаунт, з форматуванням"""
    print("🔐 Авторизація сервіс-акаунтом Google...")
    creds = get_google_creds()
    gc = gspread.authorize(creds)
    sh = gc.open_by_url(GS_SHEET_URL)

    try:
        worksheet = sh.worksheet(GS_WORKSHEET_NAME)
    except gspread.exceptions.WorksheetNotFound:
        worksheet = sh.add_worksheet(title=GS_WORKSHEET_NAME, rows="100", cols="20")

    print("🧹 Очищення старих даних та фільтрів...")
    worksheet.clear()
    worksheet.clear_basic_filter()

    print("✍️ Запис нових даних у таблицю...")
    set_with_dataframe(worksheet, df)

    worksheet.format("1:1", {
        "backgroundColor": {"red": 0.85, "green": 0.92, "blue": 0.83},
        "textFormat": {"bold": True},
        "horizontalAlignment": "CENTER"
    })
    worksheet.freeze(rows=1)

    last_col_letter = gspread.utils.rowcol_to_a1(1, len(df.columns))[:-1]
    filter_range = f"A1:{last_col_letter}{len(df) + 1}"
    worksheet.set_basic_filter(filter_range)

    print(f"✅ Успішно записано {len(df)} рядків!")

def main():
    print("🚀 Старт процесу...")

    token = get_graph_token()
    if not token:
        return

    file_path, original_filename = download_xml_via_graph(token)
    if not file_path:
        print("🤷 Процес зупинено: файл не завантажено.")
        return

    df = parse_xml_to_dataframe(file_path)
    if df.empty:
        print("⚠️ Дані не знайдені в XML.")
        return

    upload_to_google_sheets(df)

if __name__ == "__main__":
    main()

🚀 Старт процесу...
🔑 Отримуємо токен доступу від Azure AD...
⏳ Стукаємось до файлу: postachannya_latest.xml...
📥 Успішно завантажено: postachannya_latest.xml
🤖 Авто-скан: знайдено таблицю за шляхом [shop ➡️ orders ➡️ order] (Рядків: 38374)
🔄 Опрацьовано файл: postachannya_latest.xml
📦 Розгортаємо вкладений список у колонці 'products'...
🛠 Конвертуємо типи колонок...
🔐 Авторизація сервіс-акаунтом Google...
🧹 Очищення старих даних та фільтрів...
✍️ Запис нових даних у таблицю...


ConnectionError: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host'))

In [ ]:
# Постачання 14д
import os
import tempfile
import pandas as pd
import xmltodict
import gspread
import gspread.utils
import requests
from gspread_dataframe import set_with_dataframe
import numpy as np
from datetime import datetime, timedelta

# Авторизація через сервіс-акаунт Google (без інтерактивного логіну — працює headless)
from google.oauth2.service_account import Credentials as ServiceAccountCredentials

# Шлях до JSON-ключа сервіс-акаунта. Файл лежить ПОЗА цим репозиторієм/ноутбуком — не комітити нікуди.
# Таблицю треба розшарити (Editor) на client_email з цього ключа.
SERVICE_ACCOUNT_FILE = r"C:\Users\Administrator\.secrets\google\lbc-infrastructure-kestra-sheets-bot.json"
SCOPES = ["https://www.googleapis.com/auth/spreadsheets"]

def get_google_creds():
    return ServiceAccountCredentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=SCOPES)

# ==========================================
# ⚙️ НАЛАШТУВАННЯ MICROSOFT GRAPH API
# ==========================================
TENANT_ID = "vyriyllc.onmicrosoft.com"
CLIENT_ID = "7dd77777-084a-4585-a1fd-5f37696b4f9f"
# .env лежить поруч із ноутбуком (Colab/.env), у git не потрапляє
from dotenv import load_dotenv
load_dotenv(r"C:\Users\Administrator\Documents\DWH\Kestra YAML\Colab\.env")
CLIENT_SECRET = os.environ["MS_GRAPH_CLIENT_SECRET"]

SITE_ID = "vyriyllc.sharepoint.com,1cea33e3-a302-4a52-a75a-ff85fde14516,af520fba-93b7-4556-954d-decd4a92a686"
ITEM_PATH = "/LatestXML/Postach latest/postachannya_latest.xml"

# ==========================================
# ⚙️ НАЛАШТУВАННЯ GOOGLE SHEETS
# ==========================================
GS_SHEET_URL = "https://docs.google.com/spreadsheets/d/1yHi1I3F9qhm-V1AdqaklhwO8DZGjlqEkHwqDt35QzH8/edit?gid=1733056911#gid=1733056911"
GS_WORKSHEET_NAME = "Постачання 14д"


def download_xml_via_graph():
    """Завантаження файлу безпосередньо через Microsoft Graph API"""
    print("🔑 Отримуємо токен доступу від Azure AD...")

    token_url = f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token"
    token_data = {
        'grant_type': 'client_credentials',
        'client_id': CLIENT_ID,
        'client_secret': CLIENT_SECRET,
        'scope': 'https://graph.microsoft.com/.default'
    }

    token_res = requests.post(token_url, data=token_data)
    if token_res.status_code != 200:
        print(f"❌ Помилка авторизації:\n{token_res.text}")
        return []

    token = token_res.json().get('access_token')

    file_name = os.path.basename(ITEM_PATH)
    print(f"⏳ Стукаємось до файлу: {file_name}...")

    download_url = f"https://graph.microsoft.com/v1.0/sites/{SITE_ID}/drive/root:{ITEM_PATH}:/content"
    headers = {'Authorization': f'Bearer {token}'}

    file_res = requests.get(download_url, headers=headers)
    if file_res.status_code != 200:
        print(f"❌ Помилка завантаження файлу:\n{file_res.text}")
        return []

    temp_dir = tempfile.mkdtemp()
    download_path = os.path.join(temp_dir, file_name)

    with open(download_path, "wb") as f:
        f.write(file_res.content)

    print(f"📥 Успішно завантажено: {file_name}")
    return [download_path]

def parse_xml_to_dataframe(file_paths):
    """Інтелектуальний парсинг XML у плоску таблицю з розгортанням вкладених списків"""
    all_records = []

    def find_data_records(data):
        found_lists = []

        def traverse(node, path):
            if isinstance(node, dict):
                for key, value in node.items():
                    traverse(value, path + [key])
            elif isinstance(node, list):
                if len(node) > 0 and isinstance(node[0], dict):
                    found_lists.append((path, node))

        traverse(data, [])

        if found_lists:
            best_path, best_data = max(found_lists, key=lambda x: len(x[1]))
            print(f"🤖 Авто-скан: знайдено таблицю за шляхом [{' ➡️ '.join(best_path)}] (Рядків: {len(best_data)})")
            return best_data

        def get_deepest(node, path):
            if isinstance(node, dict) and len(node) == 1:
                k = list(node.keys())[0]
                if isinstance(node[k], (dict, list)):
                    return get_deepest(node[k], path + [k])
            return path, node

        path, node = get_deepest(data, [])
        print(f"🤖 Авто-скан: знайдено єдиний запис за шляхом [{' ➡️ '.join(path)}]")

        if isinstance(node, dict):
            return [node]
        elif isinstance(node, list):
            return node

        return [data]

    for file_path in file_paths:
        with open(file_path, 'r', encoding='utf-8') as f:
            xml_content = f.read()

        doc = xmltodict.parse(xml_content)
        records = find_data_records(doc)
        all_records.extend(records)
        print(f"🔄 Опрацьовано файл: {os.path.basename(file_path)}")

    # Початкове розгортання
    df = pd.json_normalize(all_records)

    # --- 🪄 НОВА МАГІЯ ДЛЯ РОЗГОРТАННЯ СПИСКІВ (ТОВАРІВ) ---
    # Шукаємо всі колонки, де зустрічаються списки
    list_columns = [col for col in df.columns if df[col].apply(lambda x: isinstance(x, list)).any()]

    for col in list_columns:
        print(f"📦 Знайдено вкладений список у колонці '{col}'. Розгортаємо...")
        # 1. Розбиваємо 1 рядок зі списком на N рядків (explode)
        df = df.explode(col).reset_index(drop=True)

        # 2. Якщо всередині списку лежать словники (ваші товари), робимо з них окремі колонки
        if df[col].apply(lambda x: isinstance(x, dict)).any():
            # Перетворюємо вміст на табличку (якщо там не словник, ставимо порожній {})
            col_df = pd.json_normalize(df[col].apply(lambda x: x if isinstance(x, dict) else {}))

            # Додаємо префікс колонки, щоб уникнути дублювання імен (наприклад, product_product_name)
            col_df = col_df.add_prefix(f"{col}_")

            # Приєднуємо до основної таблиці та видаляємо стару колонку зі словниками
            df = df.drop(columns=[col]).join(col_df)

    return df

import pandas as pd
from datetime import datetime, timedelta
import numpy as np

def apply_14days_supply_logic(df):
    """
    Відтворює логіку GAS: фільтр за 14 днів, перевірка статусів,
    заповнення відсутніх дат за order_id та дедуплікація.
    """
    if df.empty:
        return df

    # Константи з вашого скрипта
    STATUS_IDS = ["86", "52", "87", "53", "54", "412", "65"]
    ENTITY_IDS = ["1", "8", "4", "5", "54"]
    DAYS = 14

    # Робимо копію, щоб не зламати оригінал
    df = df.copy()

    # Переконуємось, що всі потрібні колонки існують (якщо ні — створюємо порожні)
    required_cols = [
        "order_id", "order_date", "order_status", "order_status_id",
        "legal_entity_name", "legal_entity_id", "product_id", "product_name",
        "product_delivery_date", "product_left"
    ]
    for col in required_cols:
        if col not in df.columns:
            df[col] = ""

    # Очищуємо ідентифікатори від пробілів
    df['order_id'] = df['order_id'].astype(str).str.strip()
    df['order_status_id'] = df['order_status_id'].astype(str).str.strip()
    df['legal_entity_id'] = df['legal_entity_id'].astype(str).str.strip()
    df['product_id'] = df['product_id'].astype(str).str.strip()

    # --- 1. КЕШУВАННЯ ДАТ (аналог orderDatesCache) ---
    # Перетворюємо дати в datetime об'єкти
    df['order_date_dt'] = pd.to_datetime(df['order_date'], errors='coerce')
    df['delivery_date_dt'] = pd.to_datetime(df['product_delivery_date'], errors='coerce')

    # Групуємо по order_id і "розмазуємо" знайдені дати на всі рядки цього замовлення
    df['order_date_dt'] = df.groupby('order_id')['order_date_dt'].transform(lambda x: x.bfill().ffill())
    df['delivery_date_dt'] = df.groupby('order_id')['delivery_date_dt'].transform(lambda x: x.bfill().ffill())

    # --- 2. ФІЛЬТРАЦІЯ (Allowed Statuses & Entities & Inventory) ---
    df = df[df['order_status_id'].isin(STATUS_IDS)]
    df = df[df['legal_entity_id'].isin(ENTITY_IDS)]

    # product_left не повинен бути порожнім
    df['product_left_clean'] = df['product_left'].astype(str).str.strip()
    df = df[df['product_left_clean'].replace(['', 'nan', 'None'], pd.NA).notna()]

    # --- 3. ЛОГІКА ДАТ ТА ВІКНА 14 ДНІВ ---
    # definitiveTargetDate = currentDeliveryDate || currentOrderDate
    df['definitive_date'] = df['delivery_date_dt'].combine_first(df['order_date_dt'])
    df = df[df['definitive_date'].notna()] # Відкидаємо, якщо дати взагалі немає

    # Розрахунок вікна: сьогоднішня північ і -14 днів
    now = datetime.now()
    today_start = now.replace(hour=0, minute=0, second=0, microsecond=0)
    window_start = today_start - timedelta(days=DAYS)

    # Фільтруємо: windowStart <= definitive_date < todayStart
    df = df[(df['definitive_date'] >= window_start) & (df['definitive_date'] < today_start)]

    # --- 4. ДЕДУПЛІКАЦІЯ (аналог deduplicationMap) ---
    # Унікальний ключ: orderId + productId + definitiveTargetDate
    df = df.drop_duplicates(subset=['order_id', 'product_id', 'definitive_date'])

    # --- 5. ПІДГОТОВКА ФІНАЛЬНИХ КОЛОНОК ---
    df['out_delivery_date'] = df['definitive_date'].dt.strftime('%Y-%m-%d %H:%M:%S')
    df['out_order_date'] = df['order_date_dt'].dt.strftime('%Y-%m-%d %H:%M:%S').fillna("")

    # Очищення product_left до формату числа з крапкою (аналог FILTER14_toNum_)
    df['out_product_left'] = df['product_left_clean'].str.replace(r'\s+', '', regex=True).str.replace(',', '.')
    df['out_product_left'] = pd.to_numeric(df['out_product_left'], errors='coerce')

    # Формуємо таблицю тільки з потрібними колонками в правильному порядку
    final_cols = [
        "order_id", "out_order_date", "order_status", "legal_entity_name",
        "product_id", "product_name", "out_delivery_date", "out_product_left"
    ]
    df_final = df[final_cols].copy()

    # Перейменовуємо назад для вивантаження
    df_final.columns = [
        "order_id", "order_date", "order_status", "legal_entity_name",
        "product_id", "product_name", "product_delivery_date", "product_left"
    ]

    # --- 6. СОРТУВАННЯ ---
    # Сортування за 7 колонкою (product_delivery_date) descending

    print(f"✅ Логіка 14-днів застосована. Залишилось рядків: {len(df_final.sort_values(by="product_delivery_date", ascending=False))}")
    return df_final.sort_values(by="product_delivery_date", ascending=False)

def upload_to_google_sheets(df):
    """Запис DataFrame у Google Sheets через сервіс-акаунт, з форматуванням"""
    print("🔐 Авторизація сервіс-акаунтом Google...")

    creds = get_google_creds()

    # Підключаємо gspread за допомогою отриманих прав
    gc = gspread.authorize(creds)
    sh = gc.open_by_url(GS_SHEET_URL)

    try:
        worksheet = sh.worksheet(GS_WORKSHEET_NAME)
    except gspread.exceptions.WorksheetNotFound:
        worksheet = sh.add_worksheet(title=GS_WORKSHEET_NAME, rows="100", cols="20")

    print("🧹 Очищення старих даних та фільтрів...")
    worksheet.clear()
    worksheet.clear_basic_filter() # Знімаємо старий фільтр, якщо був

    print("✍️ Запис нових даних у таблицю...")
    set_with_dataframe(worksheet, df)

    print("🎨 Застосування магії форматування...")
    # 1. Форматуємо перший рядок (колір, жирний шрифт, вирівнювання по центру)
    worksheet.format("1:1", {
        "backgroundColor": {
            "red": 0.85,    # Світло-зелений колір (приблизно як у вашому файлі)
            "green": 0.92,
            "blue": 0.83
        },
        "textFormat": {
            "bold": True
        },
        "horizontalAlignment": "CENTER"
    })

    # 2. Закріплюємо перший рядок
    worksheet.freeze(rows=1)

    # 3. Додаємо автофільтр на весь масив даних
    # Використовуємо глобально імпортований gspread.utils
    last_col_letter = gspread.utils.rowcol_to_a1(1, len(df.columns))[:-1]

    # Встановлюємо фільтр від A1 до останньої заповненої клітинки
    filter_range = f"A1:{last_col_letter}{len(df) + 1}"
    worksheet.set_basic_filter(filter_range)

    print(f"✅ Успішно записано {len(df)} рядків!")

def main():
    print("🚀 Старт процесу...")

    xml_files = download_xml_via_graph()
    if not xml_files:
        print("🤷 Процес зупинено: файл не завантажено.")
        return

    df = parse_xml_to_dataframe(xml_files)
    if df.empty:
        print("⚠️ Дані не знайдені в XML.")
        return

    # 🌉 НАШ "ПЕРЕХІДНИК" (Спрощений)
    # Беремо тільки розгорнуті колонки з підкресленням, щоб не створювати дублікатів!
    rename_map = {
        "products_product_id": "product_id",
        "products_product_name": "product_name",
        "products_product_delivery_date": "product_delivery_date",
        "products_product_left": "product_left"
    }

    # Перейменовуємо колонки
    df = df.rename(columns=rename_map)

    # 🪄 Застосовуємо логіку 14 днів
    df = apply_14days_supply_logic(df)

    if df.empty:
        print("⚠️ Після застосування фільтру 14 днів даних не залишилось.")
        return

    upload_to_google_sheets(df)

if __name__ == "__main__":
    main()


🚀 Старт процесу...
🔑 Отримуємо токен доступу від Azure AD...
⏳ Стукаємось до файлу: postachannya_latest.xml...
📥 Успішно завантажено: postachannya_latest.xml
🤖 Авто-скан: знайдено таблицю за шляхом [shop ➡️ orders ➡️ order] (Рядків: 38361)
🔄 Опрацьовано файл: postachannya_latest.xml
📦 Знайдено вкладений список у колонці 'products'. Розгортаємо...
✅ Логіка 14-днів застосована. Залишилось рядків: 182
🔐 Авторизація сервіс-акаунтом Google...
🧹 Очищення старих даних та фільтрів...
✍️ Запис нових даних у таблицю...
🎨 Застосування магії форматування...
✅ Успішно записано 182 рядків!


In [ ]:
# Постачання 30д
import os
import tempfile
import pandas as pd
import xmltodict
import gspread
import gspread.utils
import requests
from gspread_dataframe import set_with_dataframe
import numpy as np
from datetime import datetime, timedelta

# Авторизація через сервіс-акаунт Google (без інтерактивного логіну — працює headless)
from google.oauth2.service_account import Credentials as ServiceAccountCredentials

# Шлях до JSON-ключа сервіс-акаунта. Файл лежить ПОЗА цим репозиторієм/ноутбуком — не комітити нікуди.
# Таблицю треба розшарити (Editor) на client_email з цього ключа.
SERVICE_ACCOUNT_FILE = r"C:\Users\Administrator\.secrets\google\lbc-infrastructure-kestra-sheets-bot.json"
SCOPES = ["https://www.googleapis.com/auth/spreadsheets"]

def get_google_creds():
    return ServiceAccountCredentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=SCOPES)

# ==========================================
# ⚙️ НАЛАШТУВАННЯ MICROSOFT GRAPH API
# ==========================================
TENANT_ID = "vyriyllc.onmicrosoft.com"
CLIENT_ID = "7dd77777-084a-4585-a1fd-5f37696b4f9f"
# .env лежить поруч із ноутбуком (Colab/.env), у git не потрапляє
from dotenv import load_dotenv
load_dotenv(r"C:\Users\Administrator\Documents\DWH\Kestra YAML\Colab\.env")
CLIENT_SECRET = os.environ["MS_GRAPH_CLIENT_SECRET"]

SITE_ID = "vyriyllc.sharepoint.com,1cea33e3-a302-4a52-a75a-ff85fde14516,af520fba-93b7-4556-954d-decd4a92a686"
ITEM_PATH = "/LatestXML/Postach latest/postachannya_latest.xml"

# ==========================================
# ⚙️ НАЛАШТУВАННЯ GOOGLE SHEETS
# ==========================================
GS_SHEET_URL = "https://docs.google.com/spreadsheets/d/1yHi1I3F9qhm-V1AdqaklhwO8DZGjlqEkHwqDt35QzH8/edit?gid=1733056911#gid=1733056911"
GS_WORKSHEET_NAME = "Постачання 30д"


def download_xml_via_graph():
    """Завантаження файлу безпосередньо через Microsoft Graph API"""
    print("🔑 Отримуємо токен доступу від Azure AD...")

    token_url = f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token"
    token_data = {
        'grant_type': 'client_credentials',
        'client_id': CLIENT_ID,
        'client_secret': CLIENT_SECRET,
        'scope': 'https://graph.microsoft.com/.default'
    }

    token_res = requests.post(token_url, data=token_data)
    if token_res.status_code != 200:
        print(f"❌ Помилка авторизації:\n{token_res.text}")
        return []

    token = token_res.json().get('access_token')

    file_name = os.path.basename(ITEM_PATH)
    print(f"⏳ Стукаємось до файлу: {file_name}...")

    download_url = f"https://graph.microsoft.com/v1.0/sites/{SITE_ID}/drive/root:{ITEM_PATH}:/content"
    headers = {'Authorization': f'Bearer {token}'}

    file_res = requests.get(download_url, headers=headers)
    if file_res.status_code != 200:
        print(f"❌ Помилка завантаження файлу:\n{file_res.text}")
        return []

    temp_dir = tempfile.mkdtemp()
    download_path = os.path.join(temp_dir, file_name)

    with open(download_path, "wb") as f:
        f.write(file_res.content)

    print(f"📥 Успішно завантажено: {file_name}")
    return [download_path]

def parse_xml_to_dataframe(file_paths):
    """Інтелектуальний парсинг XML у плоску таблицю з розгортанням вкладених списків"""
    all_records = []

    def find_data_records(data):
        found_lists = []

        def traverse(node, path):
            if isinstance(node, dict):
                for key, value in node.items():
                    traverse(value, path + [key])
            elif isinstance(node, list):
                if len(node) > 0 and isinstance(node[0], dict):
                    found_lists.append((path, node))

        traverse(data, [])

        if found_lists:
            best_path, best_data = max(found_lists, key=lambda x: len(x[1]))
            print(f"🤖 Авто-скан: знайдено таблицю за шляхом [{' ➡️ '.join(best_path)}] (Рядків: {len(best_data)})")
            return best_data

        def get_deepest(node, path):
            if isinstance(node, dict) and len(node) == 1:
                k = list(node.keys())[0]
                if isinstance(node[k], (dict, list)):
                    return get_deepest(node[k], path + [k])
            return path, node

        path, node = get_deepest(data, [])
        print(f"🤖 Авто-скан: знайдено єдиний запис за шляхом [{' ➡️ '.join(path)}]")

        if isinstance(node, dict):
            return [node]
        elif isinstance(node, list):
            return node

        return [data]

    for file_path in file_paths:
        with open(file_path, 'r', encoding='utf-8') as f:
            xml_content = f.read()

        doc = xmltodict.parse(xml_content)
        records = find_data_records(doc)
        all_records.extend(records)
        print(f"🔄 Опрацьовано файл: {os.path.basename(file_path)}")

    # Початкове розгортання
    df = pd.json_normalize(all_records)

    # --- 🪄 НОВА МАГІЯ ДЛЯ РОЗГОРТАННЯ СПИСКІВ (ТОВАРІВ) ---
    # Шукаємо всі колонки, де зустрічаються списки
    list_columns = [col for col in df.columns if df[col].apply(lambda x: isinstance(x, list)).any()]

    for col in list_columns:
        print(f"📦 Знайдено вкладений список у колонці '{col}'. Розгортаємо...")
        # 1. Розбиваємо 1 рядок зі списком на N рядків (explode)
        df = df.explode(col).reset_index(drop=True)

        # 2. Якщо всередині списку лежать словники (ваші товари), робимо з них окремі колонки
        if df[col].apply(lambda x: isinstance(x, dict)).any():
            # Перетворюємо вміст на табличку (якщо там не словник, ставимо порожній {})
            col_df = pd.json_normalize(df[col].apply(lambda x: x if isinstance(x, dict) else {}))

            # Додаємо префікс колонки, щоб уникнути дублювання імен (наприклад, product_product_name)
            col_df = col_df.add_prefix(f"{col}_")

            # Приєднуємо до основної таблиці та видаляємо стару колонку зі словниками
            df = df.drop(columns=[col]).join(col_df)

    return df

import pandas as pd
from datetime import datetime, timedelta
import numpy as np

def _get_fallback_df():
    """Допоміжна функція: повертає системне повідомлення, якщо даних немає"""
    return pd.DataFrame([{
        "order_id": "000000",
        "order_date": "",
        "order_status": "Системне повідомлення",
        "legal_entity_name": "-",
        "product_id": "000000",
        "product_name": "Немає очікуваних поставок за фільтром",
        "product_delivery_date": "",
        "product_left": 0
    }])

def apply_30days_supply_logic(df):
    """
    Відтворює логіку GAS: фільтр за 30 днів ВПЕРЕД, перевірка статусів,
    заповнення відсутніх дат доставки та системне повідомлення при 0 рядків.
    """
    if df.empty:
        return _get_fallback_df()

    STATUS_IDS = ["86", "52", "87", "53", "54", "412", "65"]
    ENTITY_IDS = ["1", "8", "4", "5", "54"]
    DAYS_AHEAD = 30

    df = df.copy()

    # Створюємо порожні колонки, якщо їх раптом немає
    required_cols = [
        "order_id", "order_date", "order_status", "order_status_id",
        "legal_entity_name", "legal_entity_id", "product_id", "product_name",
        "product_delivery_date", "product_left"
    ]
    for col in required_cols:
        if col not in df.columns:
            df[col] = ""

    # Очищуємо ідентифікатори від пробілів
    df['order_id'] = df['order_id'].astype(str).str.strip()
    df['order_status_id'] = df['order_status_id'].astype(str).str.strip()
    df['legal_entity_id'] = df['legal_entity_id'].astype(str).str.strip()
    df['product_id'] = df['product_id'].astype(str).str.strip()

    # --- 1. КЕШУВАННЯ ТА ПАРСИНГ ДАТ ---
    df['order_date_dt'] = pd.to_datetime(df['order_date'], errors='coerce')
    df['delivery_date_dt'] = pd.to_datetime(df['product_delivery_date'], errors='coerce')

    # "Розмазуємо" знайдені дати на всі товари в рамках одного order_id
    df['order_date_dt'] = df.groupby('order_id')['order_date_dt'].transform(lambda x: x.bfill().ffill())
    df['delivery_date_dt'] = df.groupby('order_id')['delivery_date_dt'].transform(lambda x: x.bfill().ffill())

    # --- 2. ФІЛЬТРАЦІЯ (Статуси, Сутності, Залишки) ---
    df = df[df['order_status_id'].isin(STATUS_IDS)]
    df = df[df['legal_entity_id'].isin(ENTITY_IDS)]

    df['product_left_clean'] = df['product_left'].astype(str).str.strip()
    df = df[df['product_left_clean'].replace(['', 'nan', 'None'], pd.NA).notna()]

    # --- 3. ВІКНО 30 ДНІВ (ТІЛЬКИ ДЛЯ ДАТИ ДОСТАВКИ) ---
    now = datetime.now()
    today_start = now.replace(hour=0, minute=0, second=0, microsecond=0)
    window_end = today_start + timedelta(days=DAYS_AHEAD + 1)

    # Фільтруємо: todayStart <= delivery_date < windowEnd
    df = df[df['delivery_date_dt'].notna()]
    df = df[(df['delivery_date_dt'] >= today_start) & (df['delivery_date_dt'] < window_end)]

    # --- 4. ДЕДУПЛІКАЦІЯ ---
    # Ключ: orderId + productId + deliveryDate
    df = df.drop_duplicates(subset=['order_id', 'product_id', 'delivery_date_dt'])

    # Якщо після фільтрації нічого не залишилось — повертаємо заглушку
    if df.empty:
        print("⚠️ Даних за 30 днів немає. Формуємо системне повідомлення.")
        return _get_fallback_df()

    # --- 5. ФОРМАТУВАННЯ КОЛОНОК ---
    df['out_delivery_date'] = df['delivery_date_dt'].dt.strftime('%Y-%m-%d %H:%M:%S')
    df['out_order_date'] = df['order_date_dt'].dt.strftime('%Y-%m-%d %H:%M:%S').fillna("")

    df['out_product_left'] = df['product_left_clean'].str.replace(r'\s+', '', regex=True).str.replace(',', '.')
    df['out_product_left'] = pd.to_numeric(df['out_product_left'], errors='coerce')

    final_cols = [
        "order_id", "out_order_date", "order_status", "legal_entity_name",
        "product_id", "product_name", "out_delivery_date", "out_product_left"
    ]
    df_final = df[final_cols].copy()

    df_final.columns = [
        "order_id", "order_date", "order_status", "legal_entity_name",
        "product_id", "product_name", "product_delivery_date", "product_left"
    ]

    # --- 6. СОРТУВАННЯ (За зростанням / Ascending) ---

    print(f"✅ Логіка 30-днів застосована. Підготовлено рядків: {len(df_final.sort_values(by="product_delivery_date", ascending=True))}")
    return df_final.sort_values(by="product_delivery_date", ascending=True)

def upload_to_google_sheets(df):
    """Запис DataFrame у Google Sheets через сервіс-акаунт, з форматуванням"""
    print("🔐 Авторизація сервіс-акаунтом Google...")

    creds = get_google_creds()

    # Підключаємо gspread за допомогою отриманих прав
    gc = gspread.authorize(creds)
    sh = gc.open_by_url(GS_SHEET_URL)

    try:
        worksheet = sh.worksheet(GS_WORKSHEET_NAME)
    except gspread.exceptions.WorksheetNotFound:
        worksheet = sh.add_worksheet(title=GS_WORKSHEET_NAME, rows="100", cols="20")

    print("🧹 Очищення старих даних та фільтрів...")
    worksheet.clear()
    worksheet.clear_basic_filter() # Знімаємо старий фільтр, якщо був

    print("✍️ Запис нових даних у таблицю...")
    set_with_dataframe(worksheet, df)

    print("🎨 Застосування магії форматування...")
    # 1. Форматуємо перший рядок (колір, жирний шрифт, вирівнювання по центру)
    worksheet.format("1:1", {
        "backgroundColor": {
            "red": 0.85,    # Світло-зелений колір (приблизно як у вашому файлі)
            "green": 0.92,
            "blue": 0.83
        },
        "textFormat": {
            "bold": True
        },
        "horizontalAlignment": "CENTER"
    })

    # 2. Закріплюємо перший рядок
    worksheet.freeze(rows=1)

    # 3. Додаємо автофільтр на весь масив даних
    # Використовуємо глобально імпортований gspread.utils
    last_col_letter = gspread.utils.rowcol_to_a1(1, len(df.columns))[:-1]

    # Встановлюємо фільтр від A1 до останньої заповненої клітинки
    filter_range = f"A1:{last_col_letter}{len(df) + 1}"
    worksheet.set_basic_filter(filter_range)

    print(f"✅ Успішно записано {len(df)} рядків!")

def main():
    print("🚀 Старт процесу...")

    xml_files = download_xml_via_graph()
    if not xml_files:
        print("🤷 Процес зупинено: файл не завантажено.")
        return

    df = parse_xml_to_dataframe(xml_files)
    if df.empty:
        print("⚠️ Дані не знайдені в XML.")
        return

    # 🌉 НАШ "ПЕРЕХІДНИК" (Спрощений)
    # Беремо тільки розгорнуті колонки з підкресленням, щоб не створювати дублікатів!
    rename_map = {
        "products_product_id": "product_id",
        "products_product_name": "product_name",
        "products_product_delivery_date": "product_delivery_date",
        "products_product_left": "product_left"
    }

    # Перейменовуємо колонки
    df = df.rename(columns=rename_map)

    # Застосовуємо логіку 30 днів
    df = apply_30days_supply_logic(df)

    if df.empty:
        print("⚠️ Після застосування фільтру 14 днів даних не залишилось.")
        return

    upload_to_google_sheets(df)

if __name__ == "__main__":
    main()


🚀 Старт процесу...
🔑 Отримуємо токен доступу від Azure AD...
⏳ Стукаємось до файлу: postachannya_latest.xml...
📥 Успішно завантажено: postachannya_latest.xml
🤖 Авто-скан: знайдено таблицю за шляхом [shop ➡️ orders ➡️ order] (Рядків: 38361)
🔄 Опрацьовано файл: postachannya_latest.xml
📦 Знайдено вкладений список у колонці 'products'. Розгортаємо...
✅ Логіка 30-днів застосована. Підготовлено рядків: 280
🔐 Авторизація сервіс-акаунтом Google...
🧹 Очищення старих даних та фільтрів...
✍️ Запис нових даних у таблицю...
🎨 Застосування магії форматування...
✅ Успішно записано 280 рядків!


In [ ]:
# Запаси
import os
import tempfile
import pandas as pd
import xmltodict
import gspread
import gspread.utils
import requests
from gspread_dataframe import set_with_dataframe
import numpy as np
from datetime import datetime, timedelta

# Авторизація через сервіс-акаунт Google (без інтерактивного логіну — працює headless)
from google.oauth2.service_account import Credentials as ServiceAccountCredentials

# Шлях до JSON-ключа сервіс-акаунта. Файл лежить ПОЗА цим репозиторієм/ноутбуком — не комітити нікуди.
# Таблицю треба розшарити (Editor) на client_email з цього ключа.
SERVICE_ACCOUNT_FILE = r"C:\Users\Administrator\.secrets\google\lbc-infrastructure-kestra-sheets-bot.json"
SCOPES = ["https://www.googleapis.com/auth/spreadsheets"]

def get_google_creds():
    return ServiceAccountCredentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=SCOPES)

# ==========================================
# ⚙️ НАЛАШТУВАННЯ MICROSOFT GRAPH API
# ==========================================
TENANT_ID = "vyriyllc.onmicrosoft.com"
CLIENT_ID = "7dd77777-084a-4585-a1fd-5f37696b4f9f"
# .env лежить поруч із ноутбуком (Colab/.env), у git не потрапляє
from dotenv import load_dotenv
load_dotenv(r"C:\Users\Administrator\Documents\DWH\Kestra YAML\Colab\.env")
CLIENT_SECRET = os.environ["MS_GRAPH_CLIENT_SECRET"]

SITE_ID = "vyriyllc.sharepoint.com,1cea33e3-a302-4a52-a75a-ff85fde14516,af520fba-93b7-4556-954d-decd4a92a686"
ITEM_PATH = "/LatestXML/Postach latest/postachannya_latest.xml"

# ==========================================
# ⚙️ НАЛАШТУВАННЯ GOOGLE SHEETS
# ==========================================
GS_SHEET_URL = "https://docs.google.com/spreadsheets/d/1yHi1I3F9qhm-V1AdqaklhwO8DZGjlqEkHwqDt35QzH8/edit?gid=1733056911#gid=1733056911"
GS_WORKSHEET_NAME = "Запаси"


def download_xml_via_graph():
    """Завантаження файлу безпосередньо через Microsoft Graph API"""
    print("🔑 Отримуємо токен доступу від Azure AD...")

    token_url = f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token"
    token_data = {
        'grant_type': 'client_credentials',
        'client_id': CLIENT_ID,
        'client_secret': CLIENT_SECRET,
        'scope': 'https://graph.microsoft.com/.default'
    }

    token_res = requests.post(token_url, data=token_data)
    if token_res.status_code != 200:
        print(f"❌ Помилка авторизації:\n{token_res.text}")
        return []

    token = token_res.json().get('access_token')

    file_name = os.path.basename(ITEM_PATH)
    print(f"⏳ Стукаємось до файлу: {file_name}...")

    download_url = f"https://graph.microsoft.com/v1.0/sites/{SITE_ID}/drive/root:{ITEM_PATH}:/content"
    headers = {'Authorization': f'Bearer {token}'}

    file_res = requests.get(download_url, headers=headers)
    if file_res.status_code != 200:
        print(f"❌ Помилка завантаження файлу:\n{file_res.text}")
        return []

    temp_dir = tempfile.mkdtemp()
    download_path = os.path.join(temp_dir, file_name)

    with open(download_path, "wb") as f:
        f.write(file_res.content)

    print(f"📥 Успішно завантажено: {file_name}")
    return [download_path]

def parse_xml_to_dataframe(file_paths):
    """Інтелектуальний парсинг XML у плоску таблицю з розгортанням вкладених списків"""
    all_records = []

    def find_data_records(data):
        found_lists = []

        def traverse(node, path):
            if isinstance(node, dict):
                for key, value in node.items():
                    traverse(value, path + [key])
            elif isinstance(node, list):
                if len(node) > 0 and isinstance(node[0], dict):
                    found_lists.append((path, node))

        traverse(data, [])

        if found_lists:
            best_path, best_data = max(found_lists, key=lambda x: len(x[1]))
            print(f"🤖 Авто-скан: знайдено таблицю за шляхом [{' ➡️ '.join(best_path)}] (Рядків: {len(best_data)})")
            return best_data

        def get_deepest(node, path):
            if isinstance(node, dict) and len(node) == 1:
                k = list(node.keys())[0]
                if isinstance(node[k], (dict, list)):
                    return get_deepest(node[k], path + [k])
            return path, node

        path, node = get_deepest(data, [])
        print(f"🤖 Авто-скан: знайдено єдиний запис за шляхом [{' ➡️ '.join(path)}]")

        if isinstance(node, dict):
            return [node]
        elif isinstance(node, list):
            return node

        return [data]

    for file_path in file_paths:
        with open(file_path, 'r', encoding='utf-8') as f:
            xml_content = f.read()

        doc = xmltodict.parse(xml_content)
        records = find_data_records(doc)
        all_records.extend(records)
        print(f"🔄 Опрацьовано файл: {os.path.basename(file_path)}")

    # Початкове розгортання
    df = pd.json_normalize(all_records)

    # --- 🪄 НОВА МАГІЯ ДЛЯ РОЗГОРТАННЯ СПИСКІВ (ТОВАРІВ) ---
    # Шукаємо всі колонки, де зустрічаються списки
    list_columns = [col for col in df.columns if df[col].apply(lambda x: isinstance(x, list)).any()]

    for col in list_columns:
        print(f"📦 Знайдено вкладений список у колонці '{col}'. Розгортаємо...")
        # 1. Розбиваємо 1 рядок зі списком на N рядків (explode)
        df = df.explode(col).reset_index(drop=True)

        # 2. Якщо всередині списку лежать словники (ваші товари), робимо з них окремі колонки
        if df[col].apply(lambda x: isinstance(x, dict)).any():
            # Перетворюємо вміст на табличку (якщо там не словник, ставимо порожній {})
            col_df = pd.json_normalize(df[col].apply(lambda x: x if isinstance(x, dict) else {}))

            # Додаємо префікс колонки, щоб уникнути дублювання імен (наприклад, product_product_name)
            col_df = col_df.add_prefix(f"{col}_")

            # Приєднуємо до основної таблиці та видаляємо стару колонку зі словниками
            df = df.drop(columns=[col]).join(col_df)

    return df

import pandas as pd
from datetime import datetime, timedelta

def apply_stocks_cost_logic(df):
    """
    Відтворює логіку GAS buildStocksCost: вікно -60/+60 днів,
    перевірка ціни (>0), статусів та формування 8 колонок звіту.
    """
    if df.empty:
        return df

    # Константи з вашого скрипта
    STATUS_IDS = ["86", "87", "53", "54", "412", "65", "83", "55"]
    ENTITY_IDS = ["1", "8", "4", "5", "54"]
    DAYS_BACK = 60
    DAYS_AHEAD = 60

    df = df.copy()

    # Перевірка наявності колонок (і створення порожніх, якщо немає)
    required_cols = [
        "order_id", "order_date", "order_status", "order_status_id",
        "legal_entity_id", "product_id", "product_name",
        "product_delivery_date", "product_left", "product_price"
    ]
    for col in required_cols:
        if col not in df.columns:
            df[col] = ""

    # Очищуємо ідентифікатори від пробілів
    df['order_id'] = df['order_id'].astype(str).str.strip()
    df['order_status_id'] = df['order_status_id'].astype(str).str.strip()
    df['legal_entity_id'] = df['legal_entity_id'].astype(str).str.strip()
    df['product_id'] = df['product_id'].astype(str).str.strip()

    # --- 1. ФІЛЬТРАЦІЯ (Статуси та Сутності) ---
    df = df[df['order_status_id'].isin(STATUS_IDS)]
    df = df[df['legal_entity_id'].isin(ENTITY_IDS)]

    # --- 2. ОЧИЩЕННЯ ТА ФІЛЬТРАЦІЯ ЧИСЕЛ (Залишки та Ціна) ---
    # Залишки (не повинні бути порожніми)
    df['product_left_clean'] = df['product_left'].astype(str).str.strip()
    df = df[df['product_left_clean'].replace(['', 'nan', 'None'], pd.NA).notna()]
    df['out_product_left'] = pd.to_numeric(
        df['product_left_clean'].str.replace(r'\s+', '', regex=True).str.replace(',', '.'),
        errors='coerce'
    )

    # Ціна (повинна бути числом > 0)
    df['product_price_clean'] = df['product_price'].astype(str).str.strip()
    df['out_product_price'] = pd.to_numeric(
        df['product_price_clean'].str.replace(r'\s+', '', regex=True).str.replace(',', '.'),
        errors='coerce'
    )
    # Відкидаємо рядки, де ціна відсутня або <= 0
    df = df[df['out_product_price'] > 0]

    # --- 3. ЛОГІКА ДАТ ТА ВІКНА 120 ДНІВ ---
    df['delivery_date_dt'] = pd.to_datetime(df['product_delivery_date'], errors='coerce')
    df['order_date_dt'] = pd.to_datetime(df['order_date'], errors='coerce')

    now = datetime.now()
    today_start = now.replace(hour=0, minute=0, second=0, microsecond=0)
    window_start = today_start - timedelta(days=DAYS_BACK)
    window_end_excl = today_start + timedelta(days=DAYS_AHEAD + 1)

    # Фільтруємо: windowStart <= delivery_date < windowEndExcl
    df = df[df['delivery_date_dt'].notna()] # Дата доставки обов'язкова для цього звіту
    df = df[(df['delivery_date_dt'] >= window_start) & (df['delivery_date_dt'] < window_end_excl)]

    # --- 4. ПІДГОТОВКА ФІНАЛЬНОЇ ТАБЛИЦІ ---
    df['out_delivery_date'] = df['delivery_date_dt'].dt.strftime('%Y-%m-%d %H:%M:%S')
    df['out_order_date'] = df['order_date_dt'].dt.strftime('%Y-%m-%d %H:%M:%S').fillna("")

    # Зверніть увагу: legal_entity_name не включено, як і у вашому GAS-скрипті
    final_cols = [
        "order_id", "out_order_date", "order_status", "product_id",
        "product_name", "out_delivery_date", "out_product_left", "out_product_price"
    ]
    df_final = df[final_cols].copy()

    # Перейменовуємо для красивого вивантаження
    df_final.columns = [
        "order_id", "order_date", "order_status", "product_id",
        "product_name", "product_delivery_date", "product_left", "product_price"
    ]

    # --- 5. СОРТУВАННЯ ---
    # За датою доставки (за спаданням, як у GAS: ascending: false)

    print(f"✅ Логіка Вартість Запасів (120 днів) застосована. Залишилось рядків: {len(df_final.sort_values(by="product_delivery_date", ascending=False))}")
    return df_final.sort_values(by="product_delivery_date", ascending=False)

def upload_to_google_sheets(df):
    """Запис DataFrame у Google Sheets через сервіс-акаунт, з форматуванням"""
    print("🔐 Авторизація сервіс-акаунтом Google...")

    creds = get_google_creds()

    # Підключаємо gspread за допомогою отриманих прав
    gc = gspread.authorize(creds)
    sh = gc.open_by_url(GS_SHEET_URL)

    try:
        worksheet = sh.worksheet(GS_WORKSHEET_NAME)
    except gspread.exceptions.WorksheetNotFound:
        worksheet = sh.add_worksheet(title=GS_WORKSHEET_NAME, rows="100", cols="20")

    print("🧹 Очищення старих даних та фільтрів...")
    worksheet.clear()
    worksheet.clear_basic_filter() # Знімаємо старий фільтр, якщо був

    print("✍️ Запис нових даних у таблицю...")
    set_with_dataframe(worksheet, df)

    print("🎨 Застосування магії форматування...")
    # 1. Форматуємо перший рядок (колір, жирний шрифт, вирівнювання по центру)
    worksheet.format("1:1", {
        "backgroundColor": {
            "red": 0.85,    # Світло-зелений колір (приблизно як у вашому файлі)
            "green": 0.92,
            "blue": 0.83
        },
        "textFormat": {
            "bold": True
        },
        "horizontalAlignment": "CENTER"
    })

    # 2. Закріплюємо перший рядок
    worksheet.freeze(rows=1)

    # 3. Додаємо автофільтр на весь масив даних
    # Використовуємо глобально імпортований gspread.utils
    last_col_letter = gspread.utils.rowcol_to_a1(1, len(df.columns))[:-1]

    # Встановлюємо фільтр від A1 до останньої заповненої клітинки
    filter_range = f"A1:{last_col_letter}{len(df) + 1}"
    worksheet.set_basic_filter(filter_range)

    print(f"✅ Успішно записано {len(df)} рядків!")

def main():
    print("🚀 Старт процесу...")

    xml_files = download_xml_via_graph()
    if not xml_files:
        print("🤷 Процес зупинено: файл не завантажено.")
        return

    df = parse_xml_to_dataframe(xml_files)
    if df.empty:
        print("⚠️ Дані не знайдені в XML.")
        return

    # 🌉 НАШ "ПЕРЕХІДНИК" (Спрощений)
    # Зверніть увагу на відступи (всі рядки мають бути на одному рівні)
    rename_map = {
        "products_product_id": "product_id",
        "products_product_name": "product_name",
        "products_product_delivery_date": "product_delivery_date",
        "products_product_left": "product_left",
        "products_product_price": "product_price"  # 👈 Ціна для запасів
    }

    # Перейменовуємо колонки
    df = df.rename(columns=rename_map)

    # 🪄 Застосовуємо логіку ВАРТОСТІ ЗАПАСІВ (120 днів)
    df = apply_stocks_cost_logic(df)

    if df.empty:
        print("⚠️ Після застосування фільтру даних не залишилось.")
        return

    upload_to_google_sheets(df)

if __name__ == "__main__":
    main()


🚀 Старт процесу...
🔑 Отримуємо токен доступу від Azure AD...
⏳ Стукаємось до файлу: postachannya_latest.xml...
📥 Успішно завантажено: postachannya_latest.xml
🤖 Авто-скан: знайдено таблицю за шляхом [shop ➡️ orders ➡️ order] (Рядків: 38361)
🔄 Опрацьовано файл: postachannya_latest.xml
📦 Знайдено вкладений список у колонці 'products'. Розгортаємо...
✅ Логіка Вартість Запасів (120 днів) застосована. Залишилось рядків: 942
🔐 Авторизація сервіс-акаунтом Google...
🧹 Очищення старих даних та фільтрів...
✍️ Запис нових даних у таблицю...
🎨 Застосування магії форматування...
✅ Успішно записано 942 рядків!


# Залишки по складам

In [ ]:
pip install Office365-REST-Python-Client gspread gspread-dataframe pandas xmltodict google-auth python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import tempfile
import pandas as pd
import xmltodict
import gspread
import gspread.utils
import requests
from gspread_dataframe import set_with_dataframe

# Авторизація через сервіс-акаунт Google (без інтерактивного логіну — працює headless)
from google.oauth2.service_account import Credentials as ServiceAccountCredentials

# Шлях до JSON-ключа сервіс-акаунта. Файл лежить ПОЗА цим репозиторієм/ноутбуком — не комітити нікуди.
# Таблицю треба розшарити (Editor) на client_email з цього ключа.
SERVICE_ACCOUNT_FILE = r"C:\Users\Administrator\.secrets\google\lbc-infrastructure-kestra-sheets-bot.json"
SCOPES = ["https://www.googleapis.com/auth/spreadsheets"]

def get_google_creds():
    return ServiceAccountCredentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=SCOPES)

# ==========================================
# ⚙️ НАЛАШТУВАННЯ MICROSOFT GRAPH API
# ==========================================
TENANT_ID = "vyriyllc.onmicrosoft.com"
CLIENT_ID = "7dd77777-084a-4585-a1fd-5f37696b4f9f"
# .env лежить поруч із ноутбуком (Colab/.env), у git не потрапляє
from dotenv import load_dotenv
load_dotenv(r"C:\Users\Administrator\Documents\DWH\Kestra YAML\Colab\.env")
CLIENT_SECRET = os.environ["MS_GRAPH_CLIENT_SECRET"]

SITE_ID = "vyriyllc.sharepoint.com,1cea33e3-a302-4a52-a75a-ff85fde14516,af520fba-93b7-4556-954d-decd4a92a686"
ITEM_PATH = "/LatestXML/Storage_balance latest/storage_balance_latest.xml"

# ==========================================
# ⚙️ НАЛАШТУВАННЯ GOOGLE SHEETS
# ==========================================
GS_SHEET_URL = "https://docs.google.com/spreadsheets/d/1hIyUSO_FYyaM4FIc6prSahlt62qxNvXOrAUIV8kh4U0/edit?gid=0#gid=0"
GS_WORKSHEET_NAME = "Залишки (всі)"


def download_xml_via_graph():
    """Завантаження файлу безпосередньо через Microsoft Graph API"""
    print("🔑 Отримуємо токен доступу від Azure AD...")

    token_url = f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token"
    token_data = {
        'grant_type': 'client_credentials',
        'client_id': CLIENT_ID,
        'client_secret': CLIENT_SECRET,
        'scope': 'https://graph.microsoft.com/.default'
    }

    token_res = requests.post(token_url, data=token_data)
    if token_res.status_code != 200:
        print(f"❌ Помилка авторизації:\n{token_res.text}")
        return []

    token = token_res.json().get('access_token')

    file_name = os.path.basename(ITEM_PATH)
    print(f"⏳ Стукаємось до файлу: {file_name}...")

    download_url = f"https://graph.microsoft.com/v1.0/sites/{SITE_ID}/drive/root:{ITEM_PATH}:/content"
    headers = {'Authorization': f'Bearer {token}'}

    file_res = requests.get(download_url, headers=headers)
    if file_res.status_code != 200:
        print(f"❌ Помилка завантаження файлу:\n{file_res.text}")
        return []

    temp_dir = tempfile.mkdtemp()
    download_path = os.path.join(temp_dir, file_name)

    with open(download_path, "wb") as f:
        f.write(file_res.content)

    print(f"📥 Успішно завантажено: {file_name}")
    return [download_path]

def parse_xml_to_dataframe(file_paths):
    """Інтелектуальний парсинг XML у плоску таблицю з розгортанням вкладених списків"""
    all_records = []

    def find_data_records(data):
        found_lists = []

        def traverse(node, path):
            if isinstance(node, dict):
                for key, value in node.items():
                    traverse(value, path + [key])
            elif isinstance(node, list):
                if len(node) > 0 and isinstance(node[0], dict):
                    found_lists.append((path, node))

        traverse(data, [])

        if found_lists:
            best_path, best_data = max(found_lists, key=lambda x: len(x[1]))
            print(f"🤖 Авто-скан: знайдено таблицю за шляхом [{' ➡️ '.join(best_path)}] (Рядків: {len(best_data)})")
            return best_data

        def get_deepest(node, path):
            if isinstance(node, dict) and len(node) == 1:
                k = list(node.keys())[0]
                if isinstance(node[k], (dict, list)):
                    return get_deepest(node[k], path + [k])
            return path, node

        path, node = get_deepest(data, [])
        print(f"🤖 Авто-скан: знайдено єдиний запис за шляхом [{' ➡️ '.join(path)}]")

        if isinstance(node, dict):
            return [node]
        elif isinstance(node, list):
            return node

        return [data]

    for file_path in file_paths:
        with open(file_path, 'r', encoding='utf-8') as f:
            xml_content = f.read()

        doc = xmltodict.parse(xml_content)
        records = find_data_records(doc)
        all_records.extend(records)
        print(f"🔄 Опрацьовано файл: {os.path.basename(file_path)}")

    # Початкове розгортання
    df = pd.json_normalize(all_records)

    # --- 🪄 НОВА МАГІЯ ДЛЯ РОЗГОРТАННЯ СПИСКІВ (ТОВАРІВ) ---
    # Шукаємо всі колонки, де зустрічаються списки
    list_columns = [col for col in df.columns if df[col].apply(lambda x: isinstance(x, list)).any()]

    for col in list_columns:
        print(f"📦 Знайдено вкладений список у колонці '{col}'. Розгортаємо...")
        # 1. Розбиваємо 1 рядок зі списком на N рядків (explode)
        df = df.explode(col).reset_index(drop=True)

        # 2. Якщо всередині списку лежать словники (ваші товари), робимо з них окремі колонки
        if df[col].apply(lambda x: isinstance(x, dict)).any():
            # Перетворюємо вміст на табличку (якщо там не словник, ставимо порожній {})
            col_df = pd.json_normalize(df[col].apply(lambda x: x if isinstance(x, dict) else {}))

            # Додаємо префікс колонки, щоб уникнути дублювання імен (наприклад, product_product_name)
            col_df = col_df.add_prefix(f"{col}_")

            # Приєднуємо до основної таблиці та видаляємо стару колонку зі словниками
            df = df.drop(columns=[col]).join(col_df)

    return df

def upload_to_google_sheets(df):
    """Запис DataFrame у Google Sheets через сервіс-акаунт, з форматуванням"""
    print("🔐 Авторизація сервіс-акаунтом Google...")

    creds = get_google_creds()

    # Підключаємо gspread за допомогою отриманих прав
    gc = gspread.authorize(creds)
    sh = gc.open_by_url(GS_SHEET_URL)

    try:
        worksheet = sh.worksheet(GS_WORKSHEET_NAME)
    except gspread.exceptions.WorksheetNotFound:
        worksheet = sh.add_worksheet(title=GS_WORKSHEET_NAME, rows="100", cols="20")

    print("🧹 Очищення старих даних та фільтрів...")
    worksheet.clear()
    worksheet.clear_basic_filter() # Знімаємо старий фільтр, якщо був

    print("✍️ Запис нових даних у таблицю...")
    set_with_dataframe(worksheet, df)

    print("🎨 Застосування магії форматування...")
    # 1. Форматуємо перший рядок (колір, жирний шрифт, вирівнювання по центру)
    worksheet.format("1:1", {
        "backgroundColor": {
            "red": 0.85,    # Світло-зелений колір (приблизно як у вашому файлі)
            "green": 0.92,
            "blue": 0.83
        },
        "textFormat": {
            "bold": True
        },
        "horizontalAlignment": "CENTER"
    })

    # 2. Закріплюємо перший рядок
    worksheet.freeze(rows=1)

    # 3. Додаємо автофільтр на весь масив даних
    # Використовуємо глобально імпортований gspread.utils
    last_col_letter = gspread.utils.rowcol_to_a1(1, len(df.columns))[:-1]

    # Встановлюємо фільтр від A1 до останньої заповненої клітинки
    filter_range = f"A1:{last_col_letter}{len(df) + 1}"
    worksheet.set_basic_filter(filter_range)

    print(f"✅ Успішно записано {len(df)} рядків!")

def main():
    print("🚀 Старт процесу...")

    xml_files = download_xml_via_graph()
    if not xml_files:
        print("🤷 Процес зупинено: файл не завантажено.")
        return

    df = parse_xml_to_dataframe(xml_files)
    if df.empty:
        print("⚠️ Дані не знайдені в XML.")
        return

    upload_to_google_sheets(df)

if __name__ == "__main__":
    main()

🚀 Старт процесу...
🔑 Отримуємо токен доступу від Azure AD...
⏳ Стукаємось до файлу: storage_balance_latest.xml...
📥 Успішно завантажено: storage_balance_latest.xml
🤖 Авто-скан: знайдено таблицю за шляхом [shop ➡️ shop ➡️ products ➡️ product] (Рядків: 57988)
🔄 Опрацьовано файл: storage_balance_latest.xml
🔐 Авторизація сервіс-акаунтом Google...
🧹 Очищення старих даних та фільтрів...
✍️ Запис нових даних у таблицю...
🎨 Застосування магії форматування...
✅ Успішно записано 57988 рядків!


# Оприбуткуваня

In [ ]:
pip install Office365-REST-Python-Client gspread gspread-dataframe pandas xmltodict google-auth python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Оприбуткування (всі) + 2026 + 30д — одне завантаження, три вивантаження
import os
import tempfile
import pandas as pd
import numpy as np
import xmltodict
import gspread
import gspread.utils
import requests
from datetime import timedelta
from gspread_dataframe import set_with_dataframe

# Авторизація через сервіс-акаунт Google (без інтерактивного логіну — працює headless)
from google.oauth2.service_account import Credentials as ServiceAccountCredentials

# Шлях до JSON-ключа сервіс-акаунта. Файл лежить ПОЗА цим репозиторієм/ноутбуком — не комітити нікуди.
# Таблицю треба розшарити (Editor) на client_email з цього ключа.
SERVICE_ACCOUNT_FILE = r"C:\Users\Administrator\.secrets\google\lbc-infrastructure-kestra-sheets-bot.json"
SCOPES = ["https://www.googleapis.com/auth/spreadsheets"]

def get_google_creds():
    return ServiceAccountCredentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=SCOPES)

# ==========================================
# ⚙️ НАЛАШТУВАННЯ MICROSOFT GRAPH API
# ==========================================
TENANT_ID = "vyriyllc.onmicrosoft.com"
CLIENT_ID = "7dd77777-084a-4585-a1fd-5f37696b4f9f"
# .env лежить поруч із ноутбуком (Colab/.env), у git не потрапляє
from dotenv import load_dotenv
load_dotenv(r"C:\Users\Administrator\Documents\DWH\Kestra YAML\Colab\.env")
CLIENT_SECRET = os.environ["MS_GRAPH_CLIENT_SECRET"]

SITE_ID = "vyriyllc.sharepoint.com,1cea33e3-a302-4a52-a75a-ff85fde14516,af520fba-93b7-4556-954d-decd4a92a686"
# Папка містить окремі файли по роках/кварталах/місяцях — тягнемо і мерджимо ВСІ
FOLDER_PATH = "/LatestXML/Storage_operations_latest"

# ==========================================
# ⚙️ НАЛАШТУВАННЯ GOOGLE SHEETS
# ==========================================
GS_SHEET_URL = "https://docs.google.com/spreadsheets/d/1NTB_t8qmMz0jd2f9FWngk6X3O7O2uSTKhiL0PWKRYRU/edit?gid=1466911334#gid=1466911334"


def get_graph_token():
    """Отримання токена доступу від Azure AD"""
    print("🔑 Отримуємо токен доступу від Azure AD...")
    token_url = f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token"
    token_data = {
        'grant_type': 'client_credentials',
        'client_id': CLIENT_ID,
        'client_secret': CLIENT_SECRET,
        'scope': 'https://graph.microsoft.com/.default'
    }
    res = requests.post(token_url, data=token_data)
    if res.status_code != 200:
        print(f"❌ Помилка авторизації:\n{res.text}")
        return None
    return res.json().get('access_token')


def list_folder_xml_files(token):
    """Список усіх .xml файлів у папці Storage_operations_latest"""
    url = f"https://graph.microsoft.com/v1.0/sites/{SITE_ID}/drive/root:{FOLDER_PATH}:/children"
    headers = {'Authorization': f'Bearer {token}'}
    res = requests.get(url, headers=headers)
    if res.status_code != 200:
        print(f"❌ Помилка отримання списку файлів:\n{res.text}")
        return []
    items = res.json().get('value', [])
    return sorted(item['name'] for item in items if item['name'].lower().endswith('.xml'))


def download_xml_via_graph(token, file_name, max_retries=3):
    """Завантаження одного файлу з папки Storage_operations_latest, з перевіркою повноти завантаження й повторами"""
    download_url = f"https://graph.microsoft.com/v1.0/sites/{SITE_ID}/drive/root:{FOLDER_PATH}/{file_name}:/content"
    headers = {'Authorization': f'Bearer {token}'}

    for attempt in range(1, max_retries + 1):
        try:
            file_res = requests.get(download_url, headers=headers, timeout=300)
        except requests.exceptions.RequestException as e:
            print(f"⚠️ Мережева помилка при завантаженні {file_name} (спроба {attempt}/{max_retries}): {e}")
            continue

        if file_res.status_code != 200:
            print(f"❌ Помилка завантаження файлу {file_name}:\n{file_res.text}")
            return None

        # Перевіряємо, чи файл не обірвався посередині (великі файли часом не докачуються до кінця)
        expected_size = file_res.headers.get('Content-Length')
        actual_size = len(file_res.content)
        if expected_size is not None and int(expected_size) != actual_size:
            print(f"⚠️ {file_name}: завантажено не повністю ({actual_size} з {expected_size} байт). Повторюємо спробу {attempt}/{max_retries}...")
            continue

        temp_dir = tempfile.mkdtemp()
        download_path = os.path.join(temp_dir, file_name)
        with open(download_path, "wb") as f:
            f.write(file_res.content)
        return download_path

    print(f"❌ Не вдалося коректно завантажити {file_name} після {max_retries} спроб.")
    return None


def parse_xml_to_dataframe(file_path):
    """Інтелектуальний парсинг XML у плоску таблицю з розгортанням вкладених списків та фільтрацією (тільки type == incoming)"""
    all_records = []

    def find_data_records(data):
        found_lists = []

        def traverse(node, path):
            if isinstance(node, dict):
                for key, value in node.items():
                    traverse(value, path + [key])
            elif isinstance(node, list):
                if len(node) > 0 and isinstance(node[0], dict):
                    found_lists.append((path, node))

        traverse(data, [])

        if found_lists:
            best_path, best_data = max(found_lists, key=lambda x: len(x[1]))
            print(f"🤖 Авто-скан: знайдено таблицю за шляхом [{' ➡️ '.join(best_path)}] (Всього рядків: {len(best_data)})")
            return best_data

        def get_deepest(node, path):
            if isinstance(node, dict) and len(node) == 1:
                k = list(node.keys())[0]
                if isinstance(node[k], (dict, list)):
                    return get_deepest(node[k], path + [k])
            return path, node

        path, node = get_deepest(data, [])
        print(f"🤖 Авто-скан: знайдено єдиний запис за шляхом [{' ➡️ '.join(path)}]")

        if isinstance(node, dict):
            return [node]
        elif isinstance(node, list):
            return node

        return [data]

    with open(file_path, 'r', encoding='utf-8') as f:
        xml_content = f.read()

    doc = xmltodict.parse(xml_content)
    records = find_data_records(doc)

    # --- 🛑 ФІЛЬТР: Беремо тільки Оприбуткування ---
    filtered_records = []
    for op_data in records:
        if isinstance(op_data, dict):
            if op_data.get("type") != "incoming":
                continue
        filtered_records.append(op_data)

    all_records.extend(filtered_records)
    print(f"🔄 Опрацьовано файл: {os.path.basename(file_path)} (Пройшли фільтр: {len(filtered_records)} з {len(records)})")

    if not all_records:
        return pd.DataFrame()

    df = pd.json_normalize(all_records)

    list_columns = [col for col in df.columns if df[col].apply(lambda x: isinstance(x, list)).any()]
    for col in list_columns:
        print(f"📦 Знайдено вкладений список у колонці '{col}'. Розгортаємо...")
        df = df.explode(col).reset_index(drop=True)
        if df[col].apply(lambda x: isinstance(x, dict)).any():
            col_df = pd.json_normalize(df[col].apply(lambda x: x if isinstance(x, dict) else {}))
            col_df = col_df.add_prefix(f"{col}_")
            df = df.drop(columns=[col]).join(col_df)

    return df


def fetch_all_operations():
    """Завантажує та обʼєднує ВСІ файли з папки Storage_operations_latest в один DataFrame"""
    token = get_graph_token()
    if not token:
        return pd.DataFrame()

    file_names = list_folder_xml_files(token)
    print(f"📂 Знайдено {len(file_names)} файлів у папці Storage_operations_latest.")

    frames = []
    for i, file_name in enumerate(file_names, 1):
        print(f"⬇️ ({i}/{len(file_names)}) Завантажуємо {file_name}...")
        download_path = download_xml_via_graph(token, file_name)
        if not download_path:
            continue
        df_part = parse_xml_to_dataframe(download_path)
        os.remove(download_path)
        print(f"   -> {len(df_part)} рядків після фільтра")
        if not df_part.empty:
            frames.append(df_part)

    if not frames:
        return pd.DataFrame()

    df_all = pd.concat(frames, ignore_index=True, sort=False)

    if 'id' in df_all.columns:
        before = len(df_all)
        df_all = df_all.drop_duplicates(subset=['id']).reset_index(drop=True)
        print(f"🧬 Обʼєднано {before} рядків, після дедуплікації за 'id': {len(df_all)}")
    else:
        print(f"🧬 Обʼєднано {len(df_all)} рядків.")

    return df_all


def upload_to_google_sheets(df, worksheet_name):
    """Запис DataFrame у Google Sheets через сервіс-акаунт, з форматуванням"""
    print(f"✍️ Записуємо у вкладку '{worksheet_name}'...")
    creds = get_google_creds()
    gc = gspread.authorize(creds)
    sh = gc.open_by_url(GS_SHEET_URL)

    try:
        worksheet = sh.worksheet(worksheet_name)
    except gspread.exceptions.WorksheetNotFound:
        worksheet = sh.add_worksheet(title=worksheet_name, rows="100", cols="20")

    worksheet.clear()
    worksheet.clear_basic_filter()

    set_with_dataframe(worksheet, df)

    worksheet.format("1:1", {
        "backgroundColor": {"red": 0.85, "green": 0.92, "blue": 0.83},
        "textFormat": {"bold": True},
        "horizontalAlignment": "CENTER"
    })
    worksheet.freeze(rows=1)

    last_col_letter = gspread.utils.rowcol_to_a1(1, len(df.columns))[:-1]
    filter_range = f"A1:{last_col_letter}{len(df) + 1}"
    worksheet.set_basic_filter(filter_range)

    print(f"✅ '{worksheet_name}': записано {len(df)} рядків!")


def _get_warehouse_fallback():
    """Повертає системне повідомлення, якщо операцій по складах немає"""
    return pd.DataFrame([{
        "product_id": "000000",
        "product_name": "Немає операцій по обраних складах за останні 30 днів",
        "product_amount": 0,
        "warehouse_name": "-",
        "op_date": ""
    }])


def _pick_first_existing(df, *candidates):
    """Бере першу наявну колонку з переліку кандидатів (схема нестабільна: 1 товар -> products.product.X, кілька -> products.product_X)"""
    for c in candidates:
        if c in df.columns:
            return df[c]
    return pd.Series([pd.NA] * len(df), index=df.index)


def build_warehouse_input(df_all):
    """Збирає уніфіковані колонки для логіки складів із реальних (нестабільних) назв колонок"""
    product_id = _pick_first_existing(df_all, "products.product.productid").combine_first(
        _pick_first_existing(df_all, "products.product_productid"))
    product_name = _pick_first_existing(df_all, "products.product.productname").combine_first(
        _pick_first_existing(df_all, "products.product_productname"))
    product_amount = _pick_first_existing(df_all, "products.product.amount").combine_first(
        _pick_first_existing(df_all, "products.product_amount"))

    return pd.DataFrame({
        "op_id": _pick_first_existing(df_all, "id"),
        "op_date": _pick_first_existing(df_all, "date"),
        "storage_from": _pick_first_existing(df_all, "storagefromname"),
        "storage_to": _pick_first_existing(df_all, "storagetoname"),
        "product_id": product_id,
        "product_name": product_name,
        "product_amount": product_amount,
    })


def apply_warehouse_30days_logic(df):
    """
    Відтворює логіку GAS buildWarehouseStorage30Days:
    динамічне вікно 30 днів від останньої транзакції, фільтрація складів,
    визначення цільового складу та дедуплікація.
    """
    if df.empty:
        return _get_warehouse_fallback()

    allowed_warehouses = {
        "Склад матеріалів Б1", "Склад Б2 Кузня", "Склад матеріалів С1",
        "Склад матеріалів С2", "Склад матеріалів С3", "Склад матеріалів С4",
        "Склад матеріалів С5", "Склад матеріалів С6", "Склад матеріалів С7",
        "Контейнер 28", "Контейнер 40"
    }

    df = df.copy()

    required_cols = [
        "op_id", "op_date", "storage_from", "storage_to",
        "product_id", "product_name", "product_amount"
    ]
    for col in required_cols:
        if col not in df.columns:
            df[col] = ""

    for col in ["op_id", "product_id", "product_name", "storage_from", "storage_to"]:
        df[col] = df[col].astype(str).str.strip()

    mask_from = df['storage_from'].isin(allowed_warehouses)
    mask_to = df['storage_to'].isin(allowed_warehouses)
    df = df[mask_from | mask_to]

    if df.empty:
        return _get_warehouse_fallback()

    df['op_date_dt'] = pd.to_datetime(df['op_date'], errors='coerce')

    max_date = df['op_date_dt'].max()
    if pd.isna(max_date):
        print("⚠️ Не знайдено коректних дат. Формуємо заглушку.")
        return _get_warehouse_fallback()

    cutoff_date = max_date - timedelta(days=30)
    df = df[df['op_date_dt'] >= cutoff_date]

    df['product_amount_clean'] = df['product_amount'].astype(str).str.strip()
    df = df[df['product_amount_clean'].replace(['', 'nan', 'None'], pd.NA).notna()]
    df['out_product_amount'] = pd.to_numeric(
        df['product_amount_clean'].str.replace(r'\s+', '', regex=True).str.replace(',', '.'),
        errors='coerce'
    )
    df = df[df['out_product_amount'].notna()]

    df['warehouse_name'] = np.where(
        df['storage_to'].isin(allowed_warehouses),
        df['storage_to'],
        df['storage_from']
    )

    df = df.drop_duplicates(subset=['op_id', 'product_id', 'warehouse_name'])

    if df.empty:
        return _get_warehouse_fallback()

    df['out_date'] = df['op_date_dt'].dt.strftime('%Y-%m-%d %H:%M:%S').fillna("")

    final_cols = ["product_id", "product_name", "out_product_amount", "warehouse_name", "out_date"]
    df_final = df[final_cols].copy()
    df_final.columns = ["product_id", "product_name", "product_amount", "warehouse_name", "op_date"]

    df_final = df_final.sort_values(by=["warehouse_name", "product_name"], ascending=[True, True])

    print(f"✅ Логіка Оприбуткування на склади застосована. Підготовлено рядків: {len(df_final)}")
    return df_final


def main():
    print("🚀 Старт процесу: обʼєднання ВСІХ файлів Storage_operations_latest (2022–поточний)...")

    df_all = fetch_all_operations()
    if df_all.empty:
        print("⚠️ Дані не знайдені в жодному з файлів.")
        return

    # 1️⃣ "Оприбуткування (всі)" — повний обʼєднаний набір
    upload_to_google_sheets(df_all, "Оприбуткування (всі)")

    # 2️⃣ "Оприбуткування 2026" — фільтр за роком з уже завантажених даних (без повторного скачування)
    if 'date' in df_all.columns:
        df_2026 = df_all[df_all['date'].astype(str).str.startswith("2026")].reset_index(drop=True)
    else:
        df_2026 = pd.DataFrame()
    if df_2026.empty:
        print("⚠️ Дані за 2026 рік не знайдені.")
    else:
        upload_to_google_sheets(df_2026, "Оприбуткування 2026")

    # 3️⃣ "Оприбуткування 30д" — агрегація по складах за останні 30 днів з тих самих даних
    df_warehouse_input = build_warehouse_input(df_all)
    df_30d = apply_warehouse_30days_logic(df_warehouse_input)
    upload_to_google_sheets(df_30d, "Оприбуткування 30д")

if __name__ == "__main__":
    main()

🚀 Старт процесу...
🔑 Отримуємо токен доступу від Azure AD...
⏳ Стукаємось до файлу: storage_operations_2026_08_latest.xml...
📥 Успішно завантажено: storage_operations_2026_08_latest.xml
🤖 Авто-скан: знайдено таблицю за шляхом [shop ➡️ operations ➡️ operation] (Всього рядків: 16190)
🔄 Опрацьовано файл: storage_operations_2026_08_latest.xml (Пройшли фільтр: 2442 з 16190)
📦 Знайдено вкладений список у колонці 'products.product'. Розгортаємо...
🔐 Авторизація сервіс-акаунтом Google...
🧹 Очищення старих даних та фільтрів...
✍️ Запис нових даних у таблицю...
🎨 Застосування магії форматування...
✅ Успішно записано 4153 рядків!


In [ ]:
# "Оприбуткування 2026" тепер рахується всередині клітинки "Оприбуткування (всі)" вище —
# один прогін тягне й мерджить усі файли з папки, і одразу вивантажує всі 3 вкладки
# (Оприбуткування (всі) / 2026 / 30д), щоб не качати ті самі ~1.1 ГБ тричі.
# Запускати цю клітинку окремо більше не потрібно.


🚀 Старт процесу...
🔑 Отримуємо токен доступу від Azure AD...
⏳ Стукаємось до файлу: storage_operations_2026_08_latest.xml...
📥 Успішно завантажено: storage_operations_2026_08_latest.xml
🤖 Авто-скан: знайдено таблицю за шляхом [shop ➡️ operations ➡️ operation] (Всього рядків: 16190)
🔄 Опрацьовано файл: storage_operations_2026_08_latest.xml (Пройшли фільтр: 2442 з 16190)
📦 Знайдено вкладений список у колонці 'products.product'. Розгортаємо...
🔐 Авторизація сервіс-акаунтом Google...
🧹 Очищення старих даних та фільтрів...
✍️ Запис нових даних у таблицю...
🎨 Застосування магії форматування...
✅ Успішно записано 4153 рядків!


In [ ]:
# "Оприбуткування 30д" тепер рахується всередині клітинки "Оприбуткування (всі)" вище —
# один прогін тягне й мерджить усі файли з папки, і одразу вивантажує всі 3 вкладки
# (Оприбуткування (всі) / 2026 / 30д), щоб не качати ті самі ~1.1 ГБ тричі.
# Запускати цю клітинку окремо більше не потрібно.


🚀 Старт процесу...
🔑 Отримуємо токен доступу від Azure AD...
⏳ Стукаємось до файлу: storage_operations_2026_08_latest.xml...
📥 Успішно завантажено: storage_operations_2026_08_latest.xml
🤖 Авто-скан: знайдено таблицю за шляхом [shop ➡️ operations ➡️ operation] (Всього рядків: 16190)
🔄 Опрацьовано файл: storage_operations_2026_08_latest.xml (Пройшли фільтр: 2442 з 16190)
📦 Знайдено вкладений список у колонці 'products.product'. Розгортаємо...
🔐 Авторизація сервіс-акаунтом Google...
🧹 Очищення старих даних та фільтрів...
✍️ Запис нових даних у таблицю...
🎨 Застосування магії форматування...
✅ Успішно записано 1 рядків!
